In [ ]:
# Load Tools
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score,confusion_matrix
from sklearn.metrics import classification_report
import matplotlib.pyplot as plt
from imblearn.over_sampling import SMOTE


In [ ]:
# Import Clean Dataset

clean_df = pd.read_csv('../datasets/clean_resume_data.csv')

In [ ]:
# Remove weak classes
remove_classes = ['BPO', 'APPAREL', 'AUTOMOBILE' ,'ARTS' ,'AGRICULTURE']

clean_df = clean_df[~clean_df['Category'].isin(remove_classes)]

In [ ]:
clean_df['Category'] = clean_df['Category'].replace({
    'ACCOUNTANT': 'FINANCE'
})

In [ ]:
clean_df['Category'] = clean_df['Category'].replace({
    'CONSULTANT': 'BUSINESS-DEVELOPMENT',
    'HEALTHCARE': 'FITNESS'
})

In [ ]:
clean_df['Category'] = clean_df['Category'].replace({
    'ADVOCATE': 'PUBLIC-RELATIONS'
})

In [ ]:
clean_df['Category'].value_counts()

In [ ]:
import sys
import os

# full project root path add
sys.path.append(r"C:\Users\Dell\minor_sem4")

from utils.cleaning import cleanResume
# remove noise words
noise_words = ["communication","management","skills","experience","knowledge","ability","responsible","work","team","project","education","excel"]

def remove_noise(text):
    for word in noise_words:
        text = text.replace(word, '')
    return text
clean_df['Feature'] = clean_df['Feature'].fillna('')
clean_df['Feature'] = clean_df['Feature'].apply(cleanResume)
clean_df['Feature'] = clean_df['Feature'].apply(remove_noise)

In [ ]:
# clean_df.columns = clean_df.columns.str.strip()

In [ ]:
clean_df['Category'].value_counts()


In [ ]:
import sys
import os

sys.path.append(os.path.abspath('..'))

from utils.cleaning import cleanResume

clean_df['Feature'] = clean_df['Feature'].apply(cleanResume)

In [ ]:
clean_df['Feature'] = clean_df['Feature'].apply(cleanResume)

clean_df = clean_df[clean_df['Feature'].str.len() > 50]
clean_df = clean_df.drop_duplicates(subset='Feature')
clean_df = clean_df.dropna()

In [ ]:
# bar chart to visualize the distribution of categories in the dataset

plt.figure(figsize=(15,7))
sns.countplot(x=clean_df['Category'])
plt.xticks(rotation=90)
plt.title("Distribution of Resume Categories")
plt.show()

In [ ]:
clean_df.shape

In [ ]:
# pie chart to visualize the percentage distribution of categories in the dataset

counts = clean_df['Category'].value_counts()
labels = clean_df['Category'].unique()
plt.figure(figsize=(9,8))

plt.pie(counts, labels=counts.index, autopct='%1.1f%%')
plt.title("Percentage Distribution of Resume Categories")
plt.show()

In [ ]:
clean_df.sample(5)

In [ ]:
clean_df.info()

In [ ]:
pd.isnull(clean_df).sum()

In [ ]:
clean_df.dropna(inplace=True)   

In [ ]:
clean_df.info()

In [ ]:
pd.isnull(clean_df).sum()

In [ ]:
clean_df.isnull().sum()
clean_df.dropna(inplace=True)

In [ ]:
# remove NaN

X = clean_df['Feature']
y = clean_df['Category']
# 1 Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42,stratify=y
)

In [ ]:
# TF-IDF

from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    max_features=9000,
    ngram_range=(1,2),   # big boost
    min_df=2,            # rare words हटाओ
    max_df=0.9,
    stop_words='english'
)
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

In [ ]:

# 3 SMOTE only on training data
smote = SMOTE(random_state=42, k_neighbors=3)

X_train_resampled, y_train_resampled = smote.fit_resample(
    X_train_tfidf,
    y_train
)


# 4 Check balanced training dataset
balanced_df = pd.DataFrame({'Category': y_train_resampled})

plt.figure(figsize=(10,5))
sns.countplot(x=balanced_df['Category'])
plt.xticks(rotation=90)
plt.title("Balanced Training Dataset using SMOTE")
plt.show()

In [ ]:


balanced_df = pd.DataFrame({'Category': y_train_resampled})

print(balanced_df['Category'].value_counts())

In [ ]:

from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score

# SVM Model
svm_model = LinearSVC(
    C=2,
    class_weight='balanced',
    max_iter=2000
)

# Train
svm_model.fit(X_train_resampled, y_train_resampled)

# Predict
y_pred = svm_model.predict(X_test_tfidf)

# Accuracy
accuracy = accuracy_score(y_test, y_pred)
accuracy_display = round(accuracy, 3)

print("Accuracy:", accuracy_display)
print(classification_report(y_test, y_pred))


In [ ]:
conf_matrix = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(8,7))
sns.heatmap(
    conf_matrix,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=svm_model.classes_,
    yticklabels=svm_model.classes_
)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix')
plt.show()

In [ ]:
# Save Files


import pickle

pickle.dump(svm_model, open('../model/category_model.pkl', 'wb'))
pickle.dump(tfidf, open('../model/category_vectorizer.pkl', 'wb'))

In [ ]:
from sklearn import set_config   
set_config(display='diagram')

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

pipe = Pipeline([
    ('tfidf', tfidf),
    ('model', LogisticRegression())
])

pipe